In [1]:
# Installing Packages for Project - can use % instead of ! if you have any installation issues
!pip install pandas
!pip install numpy
%pip install torch --index-url https://download.pytorch.org/whl/cpu
!pip install matplotlib
!pip install scikit-learn
%pip install polars
%pip install pyarrow

   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
    --------------------------------------- 0.3/11.0 MB ? eta -:--:--
    --------------------------------------- 0.3/11.0 MB ? eta -:--:--
    --------------------------------------- 0.3/11.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.0 MB 541.6 kB/s eta 0:00:20
   - -------------------------------------- 0.5/11.0 MB 541.6 kB/s eta 0:00:20
   - -------------------------------------- 0.5/11.0 MB 541.6 kB/s eta 0:00:20
   -- ------------------------------------- 0.8/11.0 MB 500.8 kB/s eta 0:00:21
   -- ------------------------------------- 0.8/11.0 MB 500.8 kB/s eta 0:00:21
   --- ------------------------------------ 1.0/11.0 MB 524.3 kB/s eta 0:00:19
   ---- ----------------------------------- 1.3/11.0 MB 568.6 kB/s eta 0:00:18
   ---- ----------------------------------- 1.3/11.0 MB 568.6 kB/s eta 0:00:18
   ------ --------

In [2]:
# Importing all packages into the project
import pandas as pd
import numpy as np
import polars as pl
import sklearn
import matplotlib.pyplot as plt
import torch
import pyarrow

In [ ]:
# Parser to read out all Raw Time-Series Data for each user
# Four subdirectories exist under the main raw folder so we go through all 
# Concat each file to the df, we get the Sensor & Device from the filename regex
# Also use filename regex filter to avoid the .DS_Store files
# Every line is validated like the reference parser: after stripping the trailing ";" it must have exactly 6 fields,
# an integer subject and timestamp, an activity code and finite x, y, z. Lines that fail are written to parse_errors.csv
# (file, line, reason, text) instead of crashing read_csv or silently becoming NaN.
# Source_Line keeps the 1-based raw line number of every kept row so each window can be traced back to its raw file.

import re 
from pathlib import Path

RAW_DIRECTORY = Path("/home/busing/Documents/Projects/NCP-Wearables/wisdm+smartphone+and+smartwatch+activity+and+biometrics+dataset/wisdm-dataset/raw/") #File path for dataset on device
FILE_NAME = re.compile(r"data_(\d+)_(accel|gyro)_(phone|watch)\.txt$")
INT_FIELD = r"-?\d+"

print(RAW_DIRECTORY.exists()) # Print confirmation that Notebook can find the filepath
print(len(list(RAW_DIRECTORY.rglob("*.txt")))) # Should print 204 as this is # of .txt files with raw user data

def parse_file(path, errors):
    m = FILE_NAME.search(path.name)
    # Read every raw line as one string ("\x01" never occurs in the data) so a bad line is logged rather than guessed at
    raw = pd.read_csv(path, header = None, names = ["text"], sep = "\x01", dtype = str,
                      keep_default_na = False, skip_blank_lines = False)["text"]
    body = raw.str.strip().str.rstrip(";")
    fields = body.str.split(",", expand = True).reindex(columns = range(6))
    xyz = fields[[3, 4, 5]].apply(pd.to_numeric, errors = "coerce").to_numpy(dtype = "float64")
    ints_ok = (fields[0].str.fullmatch(INT_FIELD) & fields[2].str.fullmatch(INT_FIELD)).fillna(False).astype(bool)

    reason = np.select([(body.str.count(",") + 1 != 6).to_numpy(),
                        ~ints_ok.to_numpy(),
                        fields[1].fillna("").str.len().eq(0).to_numpy(),
                        ~np.isfinite(xyz).all(axis = 1)],
                       ["field_count", "non_integer_subject_or_timestamp", "missing_activity", "non_numeric_or_nonfinite_xyz"],
                       default = "")
    bad = reason != ""
    if bad.any():
        errors.append(pd.DataFrame({"file": str(path), "line": np.flatnonzero(bad) + 1,
                                    "reason": reason[bad], "text": raw.to_numpy()[bad]}))

    keep = fields[~bad]
    df = pd.DataFrame({"Subject_ID": keep[0].astype("int64"), "Activity_Code": keep[1].astype("category"),
                       "Timestamp": keep[2].astype("int64"),
                       "x": xyz[~bad, 0].astype("float32"), "y": xyz[~bad, 1].astype("float32"), "z": xyz[~bad, 2].astype("float32"),
                       "Source_Line": (np.flatnonzero(~bad) + 1).astype("int32")})
    df["Device"], df["Sensor"] = m.group(3), m.group(2)
    return df

def load_WISDM(raw_dir=RAW_DIRECTORY):
    files = sorted(p for p in Path(raw_dir).rglob("*.txt") if FILE_NAME.search(p.name))
    errors = []
    df = pd.concat([parse_file(p, errors) for p in files], ignore_index = True)
    for c in ("Activity_Code", "Device", "Sensor"): 
        df[c] = df[c].astype("category")
    parse_errors = (pd.concat(errors, ignore_index = True) if errors
                    else pd.DataFrame(columns = ["file", "line", "reason", "text"]))
    parse_errors.to_csv("parse_errors.csv", index = False)
    print(f"lines kept: {len(df):,} | lines rejected: {len(parse_errors):,} (logged to parse_errors.csv)")
    return df[["Subject_ID", "Activity_Code", "Timestamp", "Device", "Sensor", "x", "y", "z", "Source_Line"]]

df = load_WISDM()

df.to_parquet("wisdm_raw.parquet") #Write data to parquet file so we do not need parse data with every kernel restart.


True
204
lines kept: 15,630,426 | lines rejected: 0 (logged to parse_errors.csv)


In [4]:
# Read in data from Parquet File that was created by the parser.
# Much quicker to load in data than parsing with Pandas on each kernel restart.

# df = pd.read_parquet("/home/busing/Documents/Projects/NCP-Wearables/wisdm_raw.parquet") # Path for Laptop
df = pd.read_parquet(r"C:\Users\Daniel Busing\Desktop\Personal\China-Internship\NCP-Wearables\wisdm_raw.parquet") # Path for Mini-PC

print(df.head()) # Confirm data loaded in after kernel reset

   Subject_ID Activity_Code        Timestamp Device Sensor         x  \
0        1600             A  252207666810782  phone  accel -0.364761   
1        1600             A  252207717164786  phone  accel -0.879730   
2        1600             A  252207767518790  phone  accel  2.001495   
3        1600             A  252207817872794  phone  accel  0.450623   
4        1600             A  252207868226798  phone  accel -2.164352   

           y         z  Source_Line  
0   8.793503  1.055084            1  
1   9.768784  1.016998            2  
2  11.109070  2.619156            3  
3  12.651642  0.184555            4  
4  13.928436 -4.422485            5  


In [5]:
# Setup "stream" as one device + sensor combination thus there are 4 per participant
# Stream joing Device & Sensor into one label like e.g phone_accel. 
# Since this follows the filename format it allows key to identify each indivdual file from the dataset
KEY = ["Subject_ID", "Stream"]

df["Stream"] = (df["Device"].astype(str) + "_" + df["Sensor"].astype(str)).astype("category")
df["Row_Order"] = df.groupby(KEY, observed = True).cumcount().astype("int32") # Add int counter per file, gives us something to sort by other than timestamp for segmentation

ACTIVITY = {"A": "Walking", "B": "Jogging", "C": "Stairs", "D": "Sitting", "E": "Standing",
            "F": "Typing", "G": "Brushing Teeth", "H": "Eating Soup", "I": "Eating Chips", "J": "Eating Pasta", "K": "Drinking from Cup",
            "L": "Eating Sandwich", "M": "Kicking Soccerball", "O": "Playing Catch with Tennis Ball", "P": "Dribbling Basketball", "Q": "Writing",
            "R": "Clapping", "S": "Folding Clothes"}

EXPECTED_USERS = list(range(1600, 1651))                
EXPECTED_STREAMS = ["phone_accel", "phone_gyro", "watch_accel", "watch_gyro"]
EXPECTED_CODES = list(ACTIVITY)                         

#Check we get the expected count output, should match dataset
print(f"rows: {len(df):,} | participants: {df['Subject_ID'].nunique()} | "
      f"activities: {df['Activity_Code'].nunique()} | streams: {df['Stream'].nunique()}")

rows: 15,630,426 | participants: 51 | activities: 18 | streams: 4


In [6]:
# Count by Participant, Activity & Stream

#Note observed = True ensure we are only getting the combinations that exist in the dataframe not new combinations that are not actually present. 

# Rows per participant per stream
by_user_stream = df.groupby(KEY, observed=True).size().unstack("Stream")
#Creates a table w/ participants as rows and streams as cols. 
#Values are the number of sensor readings per participant
display(by_user_stream)

display(by_user_stream.describe()) # Provides basic summary stats for each stream

# Rows per activity; Col per stream
# .rename(index=ACTIVITY) shows activity w/ name as opposed to with Activity code
by_act_stream = (df.groupby(["Activity_Code", "Stream"], observed=True).size().unstack("Stream").rename(index=ACTIVITY))
display(by_act_stream)

# Rows per participant; Cols per activity (all streams combined)
by_user_act = (df.groupby(["Subject_ID", "Activity_Code"], observed=True).size()
                 .unstack("Activity_Code"))
display(by_user_act)

Stream,phone_accel,phone_gyro,watch_accel,watch_gyro
Subject_ID,,,,
1600,64311,64247,65462,65435
1601,81457,81193,64840,64829
1602,84890,64286,64985,64950
1603,81841,81199,64865,64851
1604,64314,64243,64904,64870
1605,81281,81181,66195,66187
1606,64315,64243,64890,64854
1607,76702,76689,65830,65810
1608,96523,64249,68198,68156


Stream,phone_accel,phone_gyro,watch_accel,watch_gyro
count,51.000000,51.000000,51.000000,51.000000
mean,94203.980392,70757.549020,74059.725490,67457.686275
std,36177.855738,11300.882807,25962.081637,9352.135727
min,60771.000000,57106.000000,60156.000000,60119.000000
25%,64314.500000,64246.000000,64881.500000,64854.000000
50%,81459.000000,64336.000000,64906.000000,64871.000000
75%,112938.000000,80347.000000,67250.000000,67215.000000
max,163768.000000,128500.000000,160799.000000,129738.000000


Stream,phone_accel,phone_gyro,watch_accel,watch_gyro
Activity_Code,,,,
Walking,279817,203919,210495,192531
Jogging,268409,200252,205787,187833
Stairs,255645,197857,207312,180416
Sitting,264592,202370,213018,195050
Standing,269604,202351,216529,194103
Typing,246356,194540,205137,187175
Brushing Teeth,269609,202622,208720,190759
Eating Soup,270756,202408,209483,187057
Eating Chips,261360,197905,210048,192085


Activity_Code,A,B,C,D,E,F,G,H,I,J,K,L,M,O,P,Q,R,S
Subject_ID,,,,,,,,,,,,,,,,,,
1600,14351.0,14350.0,14351.0,14352.0,14350.0,14351.0,14349.0,14349.0,14349.0,14349.0,14352.0,14354.0,14351.0,14349.0,14352.0,14352.0,14351.0,15493.0
1601,16227.0,16227.0,16223.0,16225.0,16227.0,16225.0,16223.0,16228.0,16227.0,16225.0,16222.0,16225.0,16230.0,16234.0,16227.0,16476.0,16223.0,16225.0
1602,14349.0,14349.0,14350.0,14349.0,14542.0,14351.0,14351.0,19729.0,18730.0,19734.0,18888.0,14351.0,14349.0,14349.0,14349.0,14349.0,14350.0,15292.0
1603,16226.0,16229.0,16229.0,16478.0,16226.0,16226.0,16227.0,16227.0,16226.0,16261.0,16479.0,16228.0,16356.0,16230.0,16225.0,16225.0,16229.0,16229.0
1604,14350.0,14351.0,14351.0,14349.0,14350.0,14376.0,14350.0,14349.0,14352.0,14351.0,14351.0,14351.0,14350.0,14351.0,14350.0,14349.0,14349.0,14351.0
1605,16228.0,16226.0,16225.0,16226.0,16220.0,16225.0,16259.0,16224.0,18891.0,16227.0,16221.0,16223.0,16224.0,16226.0,16321.0,16225.0,16229.0,16224.0
1606,14349.0,14349.0,14350.0,14351.0,14350.0,14350.0,14350.0,14352.0,14350.0,14352.0,14349.0,14349.0,14349.0,14351.0,14350.0,14350.0,14351.0,14350.0
1607,16232.0,16229.0,16236.0,16227.0,16227.0,16227.0,16226.0,16230.0,16226.0,7205.0,16225.0,18177.0,16227.0,16228.0,16231.0,16227.0,16226.0,16225.0
1608,19718.0,19718.0,14352.0,14349.0,14349.0,14349.0,14351.0,14362.0,14349.0,20952.0,14348.0,19724.0,19718.0,19718.0,19718.0,14352.0,14349.0,14350.0


In [7]:
# Investigating Missing Files & Missing Activities

present = set(df[KEY].astype({"Stream": str}).drop_duplicates().itertuples(index=False, name=None))
# List Comprehension to do Loop to check for any missing files u = users s = streams
missing_files = [(u, s) for u in EXPECTED_USERS for s in EXPECTED_STREAMS if (u,s) not in present]
print(f"files present: {len(present)} / {len(EXPECTED_USERS) * len(EXPECTED_STREAMS)} ; Missing Files: {missing_files}")

# Expected - Every file has all 18 activities
combo = (df.groupby(KEY + ["Activity_Code"], observed = True).size().rename("rows").reset_index()
        .astype({"Stream": str, "Activity_Code": str}))
# Set of (participant, stream, activity) triples that actually exist in the dataframe
df_has = set(combo[["Subject_ID", "Stream", "Activity_Code"]].itertuples(index=False, name=None))

# The list comprehension builds all 51 x 4 x 18 expected triples and compares to df_has which provides us with the gaps.
missing_activities = pd.DataFrame(
    [(u, s, c, ACTIVITY[c]) for u in EXPECTED_USERS for s in EXPECTED_STREAMS for c in EXPECTED_CODES
     if (u, s, c) not in df_has],
    columns=["Subject_ID", "Stream", "Activity_Code", "Activity"])

print(f"Missing (participant, stream, activity) combinations: {len(missing_activities)}")

display(missing_activities)

files present: 204 / 204 ; Missing Files: []
Missing (participant, stream, activity) combinations: 35


,Subject_ID,Stream,Activity_Code,Activity
0,1607,phone_accel,J,Eating Pasta
1,1607,phone_gyro,J,Eating Pasta
2,1609,phone_accel,B,Jogging
3,1609,phone_gyro,B,Jogging
4,1616,phone_accel,B,Jogging
5,1616,phone_accel,F,Typing
6,1616,phone_gyro,B,Jogging
7,1616,phone_gyro,F,Typing
8,1616,watch_accel,B,Jogging
9,1616,watch_gyro,B,Jogging


In [8]:
# Investigate Duplicate Records & Timestamps

# keep = first; so that only Second and later copies of a record are flagged as duplicates

# Returns True for any row whose timestape already appeared earlier in the same file
dup_ts = df.duplicated(subset=KEY + ["Timestamp"], keep="first")
# Returns True when a whole row repeats - same activity, timestamp, x, y, z reading
dup_exact = df.duplicated(subset=KEY + ["Activity_Code", "Timestamp", "x", "y", "z"], keep="first") # identical row
# Returns True when timestamp is shared in the same file but the readings (x, y, z) are different
dup_conflict = dup_ts & ~dup_exact 

print(f"duplicate timestamps (extra copies): {dup_ts.sum():,}  ({dup_ts.mean():.2%} of rows)")
print(f"exact duplicate rows: {dup_exact.sum():,}")
print(f"same timestamp, different values: {dup_conflict.sum():,}")

# Lists which files contain duplicates and how many extra rows exist in that file.
dup_summary = (df[dup_ts].groupby(KEY, observed=True).size().rename("extra_rows").reset_index()
                 .query("extra_rows > 0"))
print(f"\nfiles affected: {len(dup_summary)} / {df.groupby(KEY, observed=True).ngroups}")
display(dup_summary)

duplicate timestamps (extra copies): 266,015  (1.70% of rows)
exact duplicate rows: 263,696
same timestamp, different values: 2,319

files affected: 6 / 204


,Subject_ID,Stream,extra_rows
0,1629,phone_accel,69673
1,1629,phone_gyro,64250
2,1629,watch_accel,64904
3,1629,watch_gyro,64869
4,1649,watch_accel,1160
5,1649,watch_gyro,1159


In [14]:
# Investigating Timestamps and resets

g = df.groupby(KEY, observed = True) #G represents one file as KEY indicates a Sensor, Device Tuple just as each file does
dt_raw = g["Timestamp"].diff() #Difference between each timestamp and Prev timestamp in file

# True on the first row of each activity block. Compares each row's activity with previous rows activity
new_run = df["Activity_Code"].ne(g["Activity_Code"].shift())  # Find where Activity Changes

# As First Row of New Activity Block occurs during file and not just first line, we need to find the boundary. We also check this point has a valid Timestamp
boundary = new_run & dt_raw.notna()

neg = dt_raw < 0 # Clock Resets

print(f"backwards steps: {neg.sum():,}")
print(f"at an activity boundary: {(neg & boundary).sum():,}")
print(f"inside an activity: {(neg & ~boundary).sum():,}")
print(f"zero steps (repeat timestamp): {(dt_raw == 0).sum():,}")

# Is every activity one contiguous block per file? (more than one block = recording replayed/split)
blocks = new_run.groupby([df["Subject_ID"], df["Stream"], df["Activity_Code"]], observed=True).sum()
print(f"\n(participant, stream, activity) with more than one block: {(blocks > 1).sum()}")
display(blocks[blocks > 1].reset_index(name="blocks").groupby(["Subject_ID", "Stream"], observed=True).size()
        .rename("activities_split").reset_index())


backwards steps: 1,457
at an activity boundary: 1,457
inside an activity: 0
zero steps (repeat timestamp): 0

(participant, stream, activity) with more than one block: 72


,Subject_ID,Stream,activities_split
0,1629,phone_accel,18
1,1629,phone_gyro,18
2,1629,watch_accel,18
3,1629,watch_gyro,18


In [9]:
# Investigating Sampling Rate we actually see in the data & Block Lenghts

d = df.loc[~dup_ts].copy() # Dropped duplicate timestamps so duplicates do not mess with results
g = d.groupby(KEY, observed = True) # as above, just on non-duplicated timestamps
d["new_run"] = d["Activity_Code"].ne(g["Activity_Code"].shift())
d["run_id"] = d.groupby(KEY, observed=True)["new_run"].cumsum() # number of activity block in each file
RUN = KEY + ["run_id"]
d["dt_ms"] = d.groupby(RUN, observed=True)["Timestamp"].diff() / 1e6 # step between samples in m/s within a block (does not measure gap between blocks/activities)

#
runs = (d.groupby(RUN, observed=True).agg(Activity_Code=("Activity_Code", "first"),
                                          n=("Timestamp", "size"), 
                                          t0=("Timestamp", "min"),
                                          t1=("Timestamp", "max")).reset_index())

runs["duration_s"]= (runs["t1"] - runs["t0"]) / 1e9
runs["sampling_rate_hz"] = (runs["n"] - 1) / runs["duration_s"]
runs["rate_class"] = (runs["sampling_rate_hz"] / 5).round() * 5  #nearest 5 Hz

print(f"activity blocks: {len(runs)}  (expected {len(EXPECTED_USERS)*4*18} if complete)")
# Summary Statistics - We should see Central tendacy close to 180s (3 mins) as described in dataset description
display(runs["duration_s"].describe().round(1).to_frame("duration_s (expected ~180)"))
# Displays how many blocks fall into each rate class per stream. Should see most around 20 Hz
display(runs.groupby(["Stream", "rate_class"], observed=True).size().unstack("rate_class", fill_value=0))


activity blocks: 3637  (expected 3672 if complete)


,duration_s (expected ~180)
count,3637.0
mean,179.9
std,4.2
min,121.8
25%,179.8
50%,179.8
75%,179.9
max,351.3


rate_class,10.0,20.0,25.0,30.0,35.0,40.0,45.0,50.0
Stream,,,,,,,,
phone_accel,0,495,166,1,3,2,3,240
phone_gyro,0,589,317,0,1,0,0,1
watch_accel,1,817,7,8,5,1,2,71
watch_gyro,1,816,73,8,5,1,2,1


In [15]:
# Activity Blocks where Length is significantly differnet (rouhgly more than 2 S.D from expected) from 3 mins

short_long_blocks = runs[(runs["duration_s"] < 170) | (runs["duration_s"] > 190)]
print(f"Number of blocks 2 standard deviations outside 3 mins: {len(short_long_blocks)}")

#Print a table of all activity blocks where the time is more than 2 S.D from 3 mins
display(short_long_blocks.assign(Activity=lambda t: t["Activity_Code"].astype(str).map(ACTIVITY))
        [KEY + ["Activity", "n", "duration_s", "sampling_rate_hz"]].round(1))

Number of blocks 2 standard deviations outside 3 mins: 4


,Subject_ID,Stream,Activity,n,duration_s,sampling_rate_hz
3541,1649,watch_accel,Kicking Soccerball,3914,351.3,11.1
3542,1649,watch_accel,Playing Catch with Tennis Ball,2441,121.8,20.0
3559,1649,watch_gyro,Kicking Soccerball,3912,351.2,11.1
3560,1649,watch_gyro,Playing Catch with Tennis Ball,2441,121.8,20.0


In [11]:
# Further Investigation of Sampling Rates outside 20 Hz

# Sampling rate per participant per stream - Median across Activity
rate_by_file = runs.groupby(KEY, observed=True)["sampling_rate_hz"].agg(["median", "min", "max"]).round(1)

# Is Hz Median outside Range of 18 Hz - 22 Hz? # Is Hz unstable? Fastest vs Slowest Block Differ by more than 10 Hz
unstable_hz = rate_by_file[(rate_by_file["median"] < 18) | (rate_by_file["median"] > 22) | 
                           (rate_by_file["max"] - rate_by_file["min"] > 10)] 

print(f"files not at a steady 20 Hz : {len(unstable_hz)} / {len(rate_by_file)}")
display(unstable_hz)

files not at a steady 20 Hz : 80 / 204


median   min   max
Subject_ID Stream                         
1601       phone_accel    25.1  25.1  26.5
           phone_gyro     25.1  25.1  25.1
1602       phone_accel    19.9  19.9  49.6
1603       phone_accel    25.1  25.1  26.5
           phone_gyro     25.1  25.1  25.1
...                        ...   ...   ...
1648       phone_gyro     24.8  24.8  24.8
1649       phone_accel    25.1  25.1  26.5
           phone_gyro     25.1  25.1  25.1
1650       phone_accel    49.6  49.6  49.7
           phone_gyro     24.8  24.8  24.8

[80 rows x 3 columns]

In [16]:
# Investigating Timestamp Gaps & Step Sizes within Blocks where Sample Rates Differ

w = d.dropna(subset=["dt_ms"])

# Loop throuhg cut-off provided cut-off sizes (m/s) for each cut-off provided, count the number of steps that longer than this.
print("steps longer than:")
for t_h in (100, 250, 500, 1000, 5000):
    print(f"{t_h: > 5} ms: {(w['dt_ms']> t_h).sum():,}")

long_gaps = (w.loc[w["dt_ms"] > 500, KEY + ["Activity_Code", "dt_ms"]]
            .assign(Activity=lambda t: t["Activity_Code"].astype(str).map(ACTIVITY)))
display(long_gaps)


# We know there are portions of the data where the sampling rate changes significantly in a stream
# investigation of whether this sampling rate change occurs within an activity block as this will affect number of samples in a window

# Get the 5th Percentile and 95th Percentile for each activity
step_q = w.groupby(RUN, observed=True)["dt_ms"].quantile([0.05, 0.95]).unstack()
# Find Activity Blocks where the 95th percentile is 2x greater than the 5th Percentile
mixed = step_q[step_q[0.95] > 2 * step_q[0.05]].reset_index()

#Print Summary and Table of this data
print(f"\nblocks with mixed step sizes (95% > 2 x 5%): {len(mixed)} / {len(step_q)}")
display(mixed.groupby(KEY, observed=True).size().rename("mixed_blocks").reset_index()
        .sort_values("mixed_blocks", ascending=False).head(15))

steps longer than:
  100 ms: 962
  250 ms: 552
  500 ms: 10
 1000 ms: 10
 5000 ms: 8


,Subject_ID,Stream,Activity_Code,dt_ms,Activity
12103811,1649,watch_accel,M,1569.807596,Kicking Soccerball
12103879,1649,watch_accel,M,56652.537154,Kicking Soccerball
12103947,1649,watch_accel,M,56655.903353,Kicking Soccerball
12104018,1649,watch_accel,M,17494.902891,Kicking Soccerball
12104183,1649,watch_accel,M,23844.253068,Kicking Soccerball
15544199,1649,watch_gyro,M,1569.807596,Kicking Soccerball
15544267,1649,watch_gyro,M,56652.537154,Kicking Soccerball
15544335,1649,watch_gyro,M,56655.903353,Kicking Soccerball
15544406,1649,watch_gyro,M,17494.902891,Kicking Soccerball
15544571,1649,watch_gyro,M,23844.253068,Kicking Soccerball



blocks with mixed step sizes (95% > 2 x 5%): 86 / 3637


,Subject_ID,Stream,mixed_blocks
14,1609,watch_gyro,4
13,1609,watch_accel,4
38,1628,phone_accel,4
3,1602,phone_accel,3
32,1625,phone_accel,3
45,1644,watch_gyro,3
44,1644,watch_accel,3
48,1647,phone_accel,3
51,1649,phone_accel,3
28,1621,watch_accel,2


In [17]:
# Sliding Window Size Exploration

WINDOW_SIZE = 10 # Seconds

# For each rate class, how many blocks there are and how many samples a X WINDOW_SIZE window would contain

tbl = (runs.groupby("rate_class")["sampling_rate_hz"].agg(blocks="size", mean_rate="mean").round(1)
       .assign(samples_per_size = lambda t: (t["mean_rate"] * WINDOW_SIZE).round(0)))
display(tbl)

,blocks,mean_rate,samples_per_size
rate_class,,,
10.0,2,11.1,111.0
20.0,2717,20.0,200.0
25.0,563,25.0,250.0
30.0,17,30.7,307.0
35.0,14,34.0,340.0
40.0,4,38.5,385.0
45.0,7,44.5,445.0
50.0,313,49.7,497.0


In [18]:
# Week 2 - Task 1: Segmenetation of Files at Activity Change, Clock Resets and Long Gaps
# Before any within segment sorting. 

# Finding Segmentation Boundaries

GAP_BREAK_S = 0.5 # Step longer than this starts a new segment
#RESET_TOLERANCE_S = 1.0 # A backwards step bigger than this is a clock reset

GAP_BREAK_NS = int(GAP_BREAK_S * 1e9) # Timestamps are in Nanoseconds so converting to Nanoseconds
#RESET_TOLERANCE_NS = int(RESET_TOLERANCE_S * 1e9)

clone_df = df.copy() # COPY DATAFRAME - We want to leave raw data untouched; can always create a new Parquet file with Segemented & Windowed Data

timestamp = clone_df["Timestamp"].to_numpy(dtype = "int64") 
subject_id = clone_df["Subject_ID"].to_numpy()
stream = clone_df["Stream"].cat.codes.to_numpy() 
activity = clone_df["Activity_Code"].cat.codes.to_numpy()

n = len(clone_df)


#Find the step between each row and the reading before
dt = np.zeros(n, dtype="int64") # Create an array of 0's the length of dataset
dt[1:] = timestamp [1:] - timestamp[:-1] #calculates the time differences between consecutive timestamps in the array

# Each flag is True on the First row of a new segment

new_file = np.ones(n, dtype=bool) # Row 0 is always a new file
new_file[1:] = (subject_id[1:] != subject_id[:-1]) | (stream[1:] != stream[:-1]) # True at any index where the current row has a different subject_id OR a different stream than the previous row.

new_activity = np.zeros(n, dtype = bool) 
new_activity[1:] = activity[1:] != activity [:-1] # If the activity code doesn't match

within_file = ~new_file
clock_reset = within_file & (dt < 0)
long_gap = within_file & (dt > GAP_BREAK_NS)

breaks = new_file | new_activity | clock_reset | long_gap # Breaks are anything that matches one of our flags

# Sum of all break flags 
clone_df["Seg_ID"] = np.cumsum(breaks) - 1 # -1 as 0 index

reason = np.select([new_file, new_activity, clock_reset, long_gap], 
                   ["new_file", "activity_change", "clock_reset", "long_gap"],
                   default = "")
print(f"segments: {clone_df['Seg_ID'].nunique()}")
print(pd.Series(reason[breaks]).value_counts().to_string())
print("(flags can overlap, raw flag count:", f"clock_reset = {clock_reset.sum()}, long_gaps = {long_gap.sum()})")

print(f"breaks from new file: {new_file.sum()}")
print(f"from activity change: {(new_activity & ~new_file).sum()}")
print(f"from clock resets: {clock_reset.sum()}")
print(f"from long gaps: {long_gap.sum()}")

segments: 3719
activity_change    3505
new_file            204
long_gap             10
(flags can overlap, raw flag count: clock_reset = 1457, long_gaps = 2058)
breaks from new file: 204
from activity change: 3505
from clock resets: 1457
from long gaps: 2058


In [19]:
# Task 1: Resolve Duplicates, log conflicts & Drop Replayed Copies - NO sorting, the original file order is kept

K = ["Seg_ID", "Timestamp"] # Duplicate = same timestamp inside the same segement
VAL = ["x", "y", "z"] 
FILE_TS = ["Subject_ID", "Stream", "Timestamp"] 
SEG_GROUP = ["Subject_ID", "Stream", "Seg_ID", "Activity_Code"]

EXCLUDE = [] # Exclusion Log 

def log_exclusion(frame, level, reason):
    """Logs excluded data items with their row/segment details and exclusion reason."""
    cols = ["Subject_ID", "Stream", "Seg_ID", "Activity_Code", "n_rows"]
    f = frame.reindex(columns=cols).copy()
    f["level"], f["reason"] = level, reason
    EXCLUDE.append(f)

processed_data = clone_df.copy()

# a) Duplicate timestamps INSIDE a segment
dup_all = processed_data.duplicated(K, keep=False) # Flag every row that is a duplicate including the first occurence, different to how we calculated before. 
dup_rows = processed_data.loc[dup_all, K + VAL] # Drop if duplicate Segment, Timestamp and Values
n_distinct = dup_rows.drop_duplicates().groupby(K).size() # Number distinct value sets per (Seg_ID, Timestamp)
conflict_keys = n_distinct[n_distinct > 1].index # Same timestamp, different value readings

# Log all conflicting duplicate rows to csv file as requested
conflict_log = dup_rows.merge(conflict_keys.to_frame(index=False), on = K)
conflict_log.to_csv("conflict_log.csv", index= False)
print(f"Within-Segment Duplicate Rows: {dup_all.sum()}")
print(f"Within-Segement Duplicate Timestamp and Segement with Conflicting Values: {len(conflict_keys)}")

# Rule for duplicates - exact duplicates we will keep the first; conflicting values keep the first in file order
dup_drop = processed_data.duplicated(K, keep = "first")
log_exclusion(processed_data[dup_drop].groupby(SEG_GROUP, observed = True).size().rename("n_rows").reset_index(), "rows", "duplicate_timestamp_in_segment")
processed_data = processed_data[~dup_drop]

# b) Same file + timestamp but different activity code
candidates = processed_data.loc[processed_data.duplicated(FILE_TS, keep = False), FILE_TS + ["Seg_ID", "Activity_Code"]].copy()
candidates["Activity_Code"] = candidates["Activity_Code"].astype(str)
candidates["n_act"] = candidates.groupby(FILE_TS, observed = True)["Activity_Code"].transform("nunique") # Calculates the number of unique activity codes for each distinct timestamp group in DataFrame and assigns result to a new column - n_act

label_conflict = candidates[candidates["n_act"] > 1]
label_conflict.drop(columns="n_act").to_csv("Label_Conflict_Log.csv", index = False)

print(f"Rows with a timestamp shared across different activities: {len(label_conflict)}")
print(label_conflict.groupby(SEG_GROUP, observed = True).size().to_string())

log_exclusion(label_conflict.groupby(SEG_GROUP, observed = True).size().rename("n_rows").reset_index(), "rows", "Label_Conflict_Shared_Timestamp")

processed_data = processed_data.drop(label_conflict.index)

def assert_strictly_increasing(frame):
    """Inside each segment the timestamps must strictly increase without any sorting"""
    ts = frame["Timestamp"].to_numpy()
    sid = frame["Seg_ID"].to_numpy()
    same_seg = sid[1:] == sid[:-1]
    bad = int((np.diff(ts)[same_seg] <= 0).sum())
    assert bad == 0, f"{bad} non-increasing timestamp steps inside segments"

assert_strictly_increasing(processed_data)
assert processed_data["Seg_ID"].is_monotonic_increasing, "Seg_ID is not sorted"

##
# c) Segment Table & Drop Segments that are exact copies

# Aggregate raw time-series data by segment identifier ('Seg_ID').
# Computes metadata, row counts, boundary timestamps (preserved as int64 nanoseconds),
# and an aggregate signal sum ('x_sum') to serve as a segment fingerprint.

seg = (processed_data.groupby("Seg_ID").agg(Subject_ID = ("Subject_ID", "first"),
                                            Stream = ("Stream", "first"), 
                                            Activity_Code = ("Activity_Code", "first"),
                                            n = ("Timestamp", "size"),
                                            t0 = ("Timestamp", "min"),
                                            t1 = ("Timestamp", "max"),
                                            x_sum = ("x", "sum")))

# Normalise categorical fiedls to strings to ensure we have consistent types for grouping
seg["Stream"] = seg["Stream"].astype(str) 
seg["Activity_Code"] = seg["Activity_Code"].astype(str)

#Define signature coloumns that uniquely identify a segments content and metadata
SIG = ["Subject_ID", "Stream", "Activity_Code", "t0", "t1", "n", "x_sum"]

#Identify duplicated segments; as above we will keep the first and flag any other occurances
dup_seg = seg.duplicated(SIG, keep = "first")
print(f" Number of Duplicated Segments to be dropped : {dup_seg.sum()}")
print(seg[dup_seg].groupby(["Subject_ID", "Stream"]).size().to_string())

log_exclusion(seg[dup_seg].reset_index().assign(n_rows = lambda t: t["n"]), "segment", "duplicated segment") #Logging 

# Removing Duplicates from DataFrame
processed_data = processed_data[~processed_data["Seg_ID"].isin(seg.index[dup_seg])]
seg = seg[~dup_seg].copy() # Prune dups from the aggregated metadata table 'Seg'

#Enum Repeated activity occurences per subject, stream and activity (informational only - pairing uses time overlap, not occ)
seg["occ"] = seg.groupby(["Subject_ID", "Stream", "Activity_Code"]).cumcount()
# Deconstruct stream indentifier into individual device and sensore components
seg[["Device", "Sensor"]] = seg["Stream"].str.split("_", expand = True)

print(f"Segment Kept: {len(seg)}")
print(f"(Subject, Stream, Activity) with > 1 occurrence: {(seg['occ'] > 0).sum()}") # Count repeated segment instances (cumcount() is 0-indexed)

Within-Segment Duplicate Rows: 0
Within-Segement Duplicate Timestamp and Segement with Conflicting Values: 0
Rows with a timestamp shared across different activities: 4638
Subject_ID  Stream       Seg_ID  Activity_Code
1649        watch_accel  2765    M                1160
                         2766    O                1160
            watch_gyro   3695    M                1159
                         3696    O                1159
 Number of Duplicated Segments to be dropped : 72
Subject_ID  Stream     
1629        phone_accel    18
            phone_gyro     18
            watch_accel    18
            watch_gyro     18
Segment Kept: 3647
(Subject, Stream, Activity) with > 1 occurrence: 10


In [20]:
# Week 2 - Task 2: Use Common Time Origin for Accelerometer and Gyro After Verifiying their Clock Relationship
# Propose within device validity rules - at least 160 samples per sensor per 10 second window, observed span >= 8s and no gaps above 0.5 s
# Check these choices on training records and freeze them. 
# Compute statisitics directly on retained samples; no interpolation.

# Accel and gyro segments are paired by TIME OVERLAP inside (Subject, Device, Activity), not by occurrence number:
# a gap or reset can split one sensor's activity into more segments than the other's, which shifts every later occ.
CLOCK_TOL_S = 0.25      # gyro/accel start and end times must agree within this. Gates each window's residuals (cell 20); pair offsets are reported
MIN_OVERLAP_FRAC = 0.5  # Overlap must cover at least half of the shorter segment, otherwise it is not the same recording

PAIR_KEY = ["Subject_ID", "Device", "Activity_Code"]
SEG_COLS = PAIR_KEY + ["Stream", "occ", "n", "t0", "t1"]

acc = seg.loc[seg["Sensor"] == "accel", SEG_COLS].reset_index().rename(columns={"Seg_ID": "seg_id_acc"})
gyr = seg.loc[seg["Sensor"] == "gyro", SEG_COLS].reset_index().rename(columns={"Seg_ID": "seg_id_gyro"})

# Every accel x gyro candidate inside the same (Subject, Device, Activity). Inner merge, so t0/t1 stay exact int64 ns
cand = acc.merge(gyr, on=PAIR_KEY, suffixes=("_acc", "_gyro"))
ov_ns = np.minimum(cand["t1_acc"], cand["t1_gyro"]) - np.maximum(cand["t0_acc"], cand["t0_gyro"])
shorter_ns = np.minimum(cand["t1_acc"] - cand["t0_acc"], cand["t1_gyro"] - cand["t0_gyro"])
cand["overlap_s"] = ov_ns / 1e9  # Subtract as int64 FIRST, then convert to seconds - keeps precision
cand["overlap_frac"] = np.where(shorter_ns > 0, ov_ns / shorter_ns.clip(lower = 1), 0.0)
cand = cand[cand["overlap_frac"] >= MIN_OVERLAP_FRAC].sort_values(["overlap_s", "seg_id_acc"], ascending = [False, True], kind = "stable")

# One-to-one, largest overlap first: a segment can belong to at most one pair
used_acc, used_gyro, keep = set(), set(), []
for i, a, g in zip(cand.index, cand["seg_id_acc"], cand["seg_id_gyro"]):
    if a not in used_acc and g not in used_gyro:
        keep.append(i); used_acc.add(a); used_gyro.add(g)
pairs = cand.loc[keep].sort_values(PAIR_KEY + ["t0_acc"]).reset_index(drop = True)
assert pairs["seg_id_acc"].is_unique and pairs["seg_id_gyro"].is_unique, "accel/gyro pairing is not one-to-one"
assert pairs[["t0_acc", "t1_acc", "t0_gyro", "t1_gyro"]].dtypes.eq("int64").all()

# Segments with no time-overlapping partner. They are excluded from the accel+gyro mode only;
# the accel-only fallback (cell 18) still windows every watch accel segment.
unpaired_acc = acc[~acc["seg_id_acc"].isin(pairs["seg_id_acc"])]
unpaired_gyro = gyr[~gyr["seg_id_gyro"].isin(pairs["seg_id_gyro"])]
log_exclusion(unpaired_acc.rename(columns = {"seg_id_acc": "Seg_ID", "n": "n_rows"}), "segment_pairing", "no_overlapping_gyro_segment")
log_exclusion(unpaired_gyro.rename(columns = {"seg_id_gyro": "Seg_ID", "n": "n_rows"}), "segment_pairing", "no_overlapping_accel_segment")

print(f"accel segments: {len(acc)} | gyro segments: {len(gyr)} | one-to-one pairs: {len(pairs)}")
print(f"unpaired accel segments: {len(unpaired_acc)} | unpaired gyro segments: {len(unpaired_gyro)}")
display(pd.concat([unpaired_acc.assign(sensor = "accel"), unpaired_gyro.assign(sensor = "gyro")])[PAIR_KEY + ["sensor", "n"]])

# Clock evidence: start/end offsets of each pair (gyro minus accel)
pairs["start_off_s"] = (pairs["t0_gyro"] - pairs["t0_acc"]) / 1e9
pairs["end_off_s"]   = (pairs["t1_gyro"] - pairs["t1_acc"]) / 1e9
pairs["tol_ok"] = (pairs["start_off_s"].abs() <= CLOCK_TOL_S) & (pairs["end_off_s"].abs() <= CLOCK_TOL_S)
# Occurrence of the PAIR inside (Subject, Device, Activity), in start-time order (replaces the cumcount-based occ)
pairs["occ"] = pairs.groupby(PAIR_KEY).cumcount()

print(f"Pairs within {CLOCK_TOL_S}s at segment level (evidence, not a gate): {int(pairs['tol_ok'].sum())} / {len(pairs)}")
clock_evidence = pairs.groupby("Device")[["start_off_s", "end_off_s"]].describe(percentiles=[.05, .5, .95]).T.round(4)
print(clock_evidence.to_string())

print("Tolerance Failures by device: ")
print(pairs.groupby("Device")["tol_ok"].agg(pairs="size", failing=lambda s: int((~s).sum())).to_string())

accel segments: 1827 | gyro segments: 1820 | one-to-one pairs: 1819
unpaired accel segments: 8 | unpaired gyro segments: 1


,Subject_ID,Device,Activity_Code,sensor,n
748,1641,phone,Q,accel,8933
758,1642,phone,J,accel,8931
795,1644,phone,L,accel,8931
1576,1637,watch,C,accel,8931
1598,1638,watch,H,accel,8931
1604,1638,watch,O,accel,8936
1628,1640,watch,C,accel,8932
1630,1640,watch,E,accel,8934
757,1642,phone,J,gyro,4463


Pairs within 0.25s at segment level (evidence, not a gate): 1617 / 1819
Device                phone     watch
start_off_s count  907.0000  912.0000
            mean     0.1975    0.0512
            std      0.2309    0.0376
            min     -0.2307   -0.1045
            5%       0.0000    0.0000
            50%      0.1511    0.0495
            95%      0.6406    0.1174
            max      1.5822    0.4454
end_off_s   count  907.0000  912.0000
            mean     0.0017   -0.0420
            std      0.0260    0.0218
            min     -0.0526   -0.2102
            5%      -0.0500   -0.0500
            50%      0.0000   -0.0499
            95%      0.0504    0.0001
            max      0.1343    0.0998
Tolerance Failures by device: 
        pairs  failing
Device                
phone     907      197
watch     912        5


In [21]:
# Find common origin per accel/gyro pair 
pairs["pair_id"] = np.arange(len(pairs))
pairs["origin_ns"] = np.minimum(pairs["t0_acc"], pairs["t0_gyro"])     # dtype = int64

# Seg_ID to origin_ns lookup, covering both the accel and gyro segment of every pair
seg_origin = pd.concat([pairs.set_index("seg_id_acc")["origin_ns"],
                        pairs.set_index("seg_id_gyro")["origin_ns"]])

assert seg_origin.index.is_unique, "A segment appears in more than one pair"

# Only the columns needed downstream - keeps the copies small (the full table is ~15M rows)
TW_COLS = ["Subject_ID", "Stream", "Activity_Code", "Seg_ID", "Timestamp", "Source_Line", "x", "y", "z"]

# Keep only rows that belong to a verified pair. Do this BEFORE mapping so there are no NaNs,
# because NaNs would silently turn the int64 origin into float64 and lose precision.
tw = processed_data.loc[processed_data["Seg_ID"].isin(seg_origin.index), TW_COLS].reset_index(drop = True)
origin = tw["Seg_ID"].map(seg_origin).astype("int64")
tw["t_s"] = (tw["Timestamp"] - origin) /1e9 
# No sorting: processed_data is in file order, grouped by Seg_ID, with strictly increasing time inside each segment (cell 15)
assert tw["Seg_ID"].is_monotonic_increasing

# Accel-only fallback rows: EVERY kept watch accel segment, each on its own origin (its first sample), as in the reference.
# Gyro is not used by the fallback, so an accel segment is never dropped because its gyro partner is missing or misaligned.
fallback_segs = seg.index[(seg["Device"] == "watch") & (seg["Sensor"] == "accel")]
fw = processed_data.loc[processed_data["Seg_ID"].isin(fallback_segs), TW_COLS].reset_index(drop = True)
fw["t_s"] = (fw["Timestamp"] - fw["Seg_ID"].map(seg["t0"]).astype("int64")) / 1e9
assert fw["Seg_ID"].is_monotonic_increasing

print(tw.head())
print(f"paired rows: {len(tw):,} | fallback watch-accel rows: {len(fw):,} in {len(fallback_segs)} segments")

   Subject_ID       Stream Activity_Code  Seg_ID        Timestamp  \
0        1600  phone_accel             A       0  252207666810782   
1        1600  phone_accel             A       0  252207717164786   
2        1600  phone_accel             A       0  252207767518790   
3        1600  phone_accel             A       0  252207817872794   
4        1600  phone_accel             A       0  252207868226798   

   Source_Line         x          y         z       t_s  
0            1 -0.364761   8.793503  1.055084  0.000000  
1            2 -0.879730   9.768784  1.016998  0.050354  
2            3  2.001495  11.109070  2.619156  0.100708  
3            4  0.450623  12.651642  0.184555  0.151062  
4            5 -2.164352  13.928436 -4.422485  0.201416  
paired rows: 15,286,170 | fallback watch-accel rows: 3,709,822 in 917 segments


In [22]:
# Windows & Per Sensor Statistics

WINDOW_S = 10.0
# Non-overlapping windows (step == window), matching the reference pipeline. A 5 s step doubles the window count,
# but adjacent windows then share half their samples, so support/query windows from one participant would not be
# independent in the personalisation experiments. Participant-level splits alone do not prevent that.
STEP_S = WINDOW_S

def segment_bounds(seg_ids):
    """Row range [first, last) of every segment inside an array that is grouped by Seg_ID"""
    first = np.flatnonzero(np.r_[True, seg_ids[1:] != seg_ids[:-1]]) #Find the starting indices of consecutive segments where values change
    last = np.r_[first[1:], len(seg_ids)]
    return dict(zip(seg_ids[first], zip(first, last)))

def window_stats(ts_seg, starts): 
    low = np.searchsorted(ts_seg, starts, side = 'left')
    high = np.searchsorted(ts_seg, starts + WINDOW_S, side = "left")
    cnt = high - low
    span = np.zeros(len(starts))
    gap = np.full(len(starts), np.inf)
    #gap/span uses samples inside the window; the span >= 8s rule is what limits empty spaces at the edges
    for i, (a,b) in enumerate (zip(low, high)): 
        if b - a >= 2: 
            s = ts_seg[a:b]
            span[i] = s[-1] - s[0]
            gap[i] = np.diff(s).max()
    return low, high, cnt, span, gap

def source_file(subject, sensor, device):
    return f"data_{subject}_{sensor}_{device}.txt"

def assert_windows_in_segment(w, seg_ids, prefix):
    """First AND last row of every non-empty window must belong to the window's own segment:
    no stale row positions and no window crossing an activity or recording boundary"""
    m = (w[f"{prefix}_n"] > 0).to_numpy()
    a, b = w.loc[m, f"{prefix}_a"].to_numpy(), w.loc[m, f"{prefix}_b"].to_numpy()
    s = w.loc[m, f"{prefix}_seg"].to_numpy()
    assert (seg_ids[a] == s).all() and (seg_ids[b - 1] == s).all(), f"stale or boundary-crossing {prefix} window rows"

# a) Paired accel + gyro windows on the common per-pair grid (row positions index into tw)
t = tw["t_s"].to_numpy()
seg_arr = tw["Seg_ID"].to_numpy()
bounds = segment_bounds(seg_arr)

parts = []
for p in pairs.itertuples():
    a0, a1 = bounds[p.seg_id_acc]
    g0, g1 = bounds[p.seg_id_gyro]
    ta, tg = t[a0:a1], t[g0:g1]
    t_end = max(ta[-1], tg[-1])
    starts = np.arange(0.0, t_end - WINDOW_S + 1e-9, STEP_S)
    if len(starts) == 0:
        continue
    la, ha, na, spa, gpa = window_stats(ta, starts)
    lg, hg, ng, spg, gpg = window_stats(tg, starts)
    parts.append(pd.DataFrame({
            "window_id": [f"{p.Subject_ID}|{p.Device}|{p.Activity_Code}|acc{p.seg_id_acc}|gyro{p.seg_id_gyro}|w{k}" for k in range(len(starts))],
            "pair_id" : p.pair_id, "Subject_ID" : p.Subject_ID, "Device": p.Device,
            "Activity_Code": p.Activity_Code, "occ": p.occ, "t_start": starts, "tol_ok": p.tol_ok,
            "acc_seg": p.seg_id_acc, "gyro_seg": p.seg_id_gyro,
            "acc_source": source_file(p.Subject_ID, "accel", p.Device), "gyro_source": source_file(p.Subject_ID, "gyro", p.Device),
            # Row positions in tw. Only valid for this tw, will need to re-run if tw changes
            "acc_a": a0 + la, "acc_b": a0 + ha, "gyro_a": g0 + lg, "gyro_b": g0 + hg, 
            "acc_n": na, "acc_span": spa, "acc_gap": gpa, "gyro_n": ng, "gyro_span": spg, "gyro_gap": gpg}))

windows = pd.concat(parts, ignore_index = True)
assert_windows_in_segment(windows, seg_arr, "acc")
assert_windows_in_segment(windows, seg_arr, "gyro")
windows["rate_class"] = ((windows["acc_n"]/ WINDOW_S / 5).round() * 5).astype(int) # acc samples/s, nearest 5 Hz

# b) Accel-only fallback windows: every watch accel segment on its OWN grid (starts at its first sample), only full
#    windows inside the segment - the reference logic. Row positions index into fw.
ft = fw["t_s"].to_numpy()
fseg_arr = fw["Seg_ID"].to_numpy()

fparts = []
for s_id, (a0, a1) in segment_bounds(fseg_arr).items():
    ts_seg = ft[a0:a1]
    starts = np.arange(0.0, ts_seg[-1] - WINDOW_S + 1e-9, STEP_S)
    if len(starts) == 0:
        continue
    lo, hi, n_, sp, gp = window_stats(ts_seg, starts)
    r = seg.loc[s_id]
    fparts.append(pd.DataFrame({
            "window_id": [f"{r.Subject_ID}|watch|{r.Activity_Code}|acc{s_id}|w{k}" for k in range(len(starts))],
            "Subject_ID": r.Subject_ID, "Device": "watch", "Activity_Code": r.Activity_Code, "occ": r.occ,
            "t_start": starts, "acc_seg": s_id, "acc_source": source_file(r.Subject_ID, "accel", "watch"),
            "acc_a": a0 + lo, "acc_b": a0 + hi, "acc_n": n_, "acc_span": sp, "acc_gap": gp}))

acc_windows = pd.concat(fparts, ignore_index = True)
assert_windows_in_segment(acc_windows, fseg_arr, "acc")

display(windows.head())
print(f"Number of candidate paired windows: {len(windows)} | candidate watch accel-only (fallback) windows: {len(acc_windows)}")

,window_id,pair_id,Subject_ID,Device,Activity_Code,occ,t_start,tol_ok,acc_seg,gyro_seg,...,acc_b,gyro_a,gyro_b,acc_n,acc_span,acc_gap,gyro_n,gyro_span,gyro_gap,rate_class
0,1600|phone|A|acc0|gyro928|w0,0,1600,phone,A,0,0.0,False,0,928,...,199,4707935,4708129,199,9.970104,0.050365,194,9.718334,0.050365,20
1,1600|phone|A|acc0|gyro928|w1,0,1600,phone,A,0,10.0,False,0,928,...,398,4708129,4708328,199,9.970084,0.050354,199,9.970084,0.050354,20
2,1600|phone|A|acc0|gyro928|w2,0,1600,phone,A,0,20.0,False,0,928,...,596,4708328,4708526,198,9.919754,0.050369,198,9.919754,0.050369,20
3,1600|phone|A|acc0|gyro928|w3,0,1600,phone,A,0,30.0,False,0,928,...,795,4708526,4708725,199,9.970086,0.050354,199,9.970086,0.050354,20
4,1600|phone|A|acc0|gyro928|w4,0,1600,phone,A,0,40.0,False,0,928,...,993,4708725,4708923,198,9.919727,0.050354,198,9.919727,0.050354,20


Number of candidate paired windows: 31041 | candidate watch accel-only (fallback) windows: 15518


In [23]:
# Validity rules, eligibility, participant split, check the rules on training records & freeze
import json

SEED = 42 # Fixed Seed so the split is reproducible
SPLIT_FRAC = {"train": 0.6, "eval": 0.2} # test = the remainder. Same fractions as the reference: 31 / 10 / 10 for 51 users
MIN_N, MIN_SPAN_S, MAX_GAP_S = 160, 8.0, 0.5 # The proposed rules
REQUIRED_CLASSES = sorted(ACTIVITY) # all 18 activities

def sensor_ok(w, p, min_n = MIN_N, min_span = MIN_SPAN_S, max_gap = MAX_GAP_S):
    """One sensor of a window passes all three rules"""
    return (w[f"{p}_n"] >= min_n) & (w[f"{p}_span"] >= min_span) & (w[f"{p}_gap"] <= max_gap)

def valid(w, min_n = MIN_N, min_span = MIN_SPAN_S, max_gap = MAX_GAP_S):
    """ A paired window is valid only if both sensors of the device pass all three rules"""
    return sensor_ok(w, "acc", min_n, min_span, max_gap) & sensor_ok(w, "gyro", min_n, min_span, max_gap)

windows["valid"] = valid(windows)
acc_windows["valid"] = sensor_ok(acc_windows, "acc")

# a) Eligibility from the full audit BEFORE the split is frozen (reference: verify usable data and required classes).
#    Usable data = rule-valid watch accel windows, which both feature modes need.
usable = acc_windows[acc_windows["valid"]]
elig = (usable.groupby("Subject_ID")["Activity_Code"]
              .agg(usable_windows = "size", n_classes = "nunique", classes = lambda s: "".join(sorted(set(s))))
              .reindex(EXPECTED_USERS))
elig["usable_windows"] = elig["usable_windows"].fillna(0).astype(int)
elig["n_classes"] = elig["n_classes"].fillna(0).astype(int)
elig["missing_classes"] = ["".join(c for c in REQUIRED_CLASSES if c not in have) for have in elig["classes"].fillna("")]
elig["eligible"] = elig["usable_windows"] > 0
eligible_users = np.array(sorted(elig.index[elig["eligible"]]))
print(f"eligible users (usable watch data): {len(eligible_users)} / {len(EXPECTED_USERS)}")

# b) Participant split on eligible users only - every device and recording of a person stays in one split
order = np.random.default_rng(SEED).permutation(eligible_users)
n_train = round(len(order) * SPLIT_FRAC["train"]) #Round will ensure that we get 31 users in the train
n_eval = round(len(order) * SPLIT_FRAC["eval"]) #Round will ensure that we get 10 users in the eval
splits = {"train": order[:n_train], "eval": order[n_train:n_train + n_eval], "test": order[n_train + n_eval:]} #Remaining 10 will go into test

train_split, eval_split, test_split = (set(splits[k]) for k in ("train", "eval", "test"))
# Ensure that no Subject_ID appears in more than one of the splits to ensure we don't have data leakage
assert not(train_split & eval_split) and not (train_split & test_split) and not (eval_split & test_split)
assert len(train_split) + len(eval_split) + len(test_split) == len(eligible_users)

# Map each subject to its split label and attach the label
split_map = {int(s): name for name, ids in splits.items() for s in ids}
elig["split"] = elig.index.map(split_map)

# Held-out users must have every class: macro-F1, per-class recall and the support/query personalisation split need it.
# Training users missing a class or two are kept and recorded in user_eligibility.csv.
held_out_gaps = elig[elig["split"].isin(["eval", "test"]) & (elig["missing_classes"] != "")]
assert held_out_gaps.empty, f"eval/test users are missing classes - review before freezing:\n{held_out_gaps}"
print("users missing classes (all must be in train):")
display(elig[elig["missing_classes"] != ""])

elig.to_csv("user_eligibility.csv")
pd.DataFrame([{"Subject_ID": int(u), "split": name, "seed": SEED} for name, ids in splits.items() for u in ids]).to_csv("split_manifest.csv", index = False)

windows["split"] = windows["Subject_ID"].map(split_map) # applying the split map and adding 
acc_windows["split"] = acc_windows["Subject_ID"].map(split_map)
print(windows.groupby("split")["Subject_ID"].nunique()) #Should return the split numbers set by SPLIT_FRAC

# c) Judge the proposed rules on TRAINING users only
train = windows[windows["split"] == "train"] # ONLY training rows are used to judge the rules
train_acc = acc_windows[acc_windows["split"] == "train"]

print(f"\nTRAIN paired windows: {len(train):,} | valid: {train['valid'].mean():.1%}")
print(f"TRAIN watch accel-only windows: {len(train_acc):,} | valid: {train_acc['valid'].mean():.1%}")
print("\npass rate by device:");      
print(train.groupby("Device")["valid"].mean().round(3).to_string())
print("\npass rate by rate class (acc samples/s):")
print(train.groupby("rate_class")["valid"].agg(windows="size", pass_rate="mean").round(3).to_string())
print("\npass rate by activity:")
print(train.groupby("Activity_Code")["valid"].mean().round(3).rename(index=ACTIVITY).to_string())

print("\nwhich rule fails (share of train windows):")
for p in ("acc", "gyro"):
    print(f"  {p}: n<{MIN_N}: {(train[f'{p}_n'] < MIN_N).mean():.3f} | "
          f"span<{MIN_SPAN_S}: {(train[f'{p}_span'] < MIN_SPAN_S).mean():.3f} | "
          f"gap>{MAX_GAP_S}: {(train[f'{p}_gap'] > MAX_GAP_S).mean():.3f}")

print("\nsensitivity (train pass rate):")
for mn, mg in [(140, 0.5), (160, 0.5), (180, 0.5), (160, 0.3), (160, 0.75)]:
    print(f"  min_n={mn}, max_gap={mg}: {valid(train, mn, MIN_SPAN_S, mg).mean():.3f}")

# Freeze: written once the output above has been reviewed. Eval/test are never used to change the rules
FROZEN_RULES = dict(window_s = WINDOW_S, step_s = STEP_S, min_n = MIN_N, min_span_s = MIN_SPAN_S, max_gap_s = MAX_GAP_S,
                    gap_break_s = GAP_BREAK_S, clock_reset = "any backwards step", clock_tol_s = CLOCK_TOL_S,
                    min_overlap_frac = MIN_OVERLAP_FRAC,
                    window_gate = "both sensors pass the rules and window start/end residuals within clock_tol_s; pair-level offsets reported only",
                    split_seed = SEED, split_fractions = SPLIT_FRAC,
                    split_sizes = {k: len(v) for k, v in splits.items()})

with open("frozen_rules.json", "w") as f:
    json.dump(FROZEN_RULES, f, indent = 2)

print("saved frozen_rules.json, split_manifest.csv and user_eligibility.csv")

eligible users (usable watch data): 51 / 51
users missing classes (all must be in train):


,usable_windows,n_classes,classes,missing_classes,eligible,split
Subject_ID,,,,,,
1616,289,17,ACDEFGHIJKLMOPQRS,B,True,train
1618,289,17,ABCDEFGHIJKLMPQRS,O,True,train
1637,293,17,ABCDEFGHIKLMOPQRS,J,True,train
1639,292,17,ABCDEFGHIJKMOPQRS,L,True,train
1642,272,16,ABDEGHIJKLMOPQRS,CF,True,train


split
eval     10
test     10
train    31
Name: Subject_ID, dtype: int64

TRAIN paired windows: 18,750 | valid: 100.0%
TRAIN watch accel-only windows: 9,407 | valid: 100.0%

pass rate by device:
Device
phone    1.0
watch    1.0

pass rate by rate class (acc samples/s):
            windows  pass_rate
rate_class                    
20            13201        1.0
25             1221        1.0
30                3        1.0
35                9        1.0
40                7        1.0
45                9        1.0
50             4244        1.0
55                5        1.0
60                3        1.0
65                1        1.0
70                4        1.0
75                5        1.0
80                2        1.0
85                8        1.0
90                7        1.0
95                3        1.0
100              18        1.0

pass rate by activity:
Activity_Code
Walking                           1.0
Jogging                           1.0
Stairs                     

In [24]:
# One-to-one accel/gyro window pairing with residuals, the feature-mode decision, the window manifest and exclusions

# Windows are built per pair (same pair_id + same start time) so pairing is one-to-one by construction
# Equal labels/row number not counted as evidence as windows being a match. 
# Also check passed windows and first/last sample times of the two windows agree within the Tolerance set by CLOCK_TOL_S

# One-to-One guard - one window row per (pair, start) and every segement belongs to only one pair
assert windows["window_id"].is_unique and acc_windows["window_id"].is_unique
assert not windows.duplicated(["pair_id", "t_start"]).any() # Check window was paired more than once
assert pairs["seg_id_acc"].is_unique and pairs["seg_id_gyro"].is_unique # A segment is in two pairs

# Check first/last observed sample times inside every non-empty window 
has = ((windows["acc_n"] > 0) & (windows["gyro_n"] > 0)).to_numpy()

def edge(a_col, b_col, arr, which): 
    a = windows.loc[has, a_col].to_numpy()
    b = windows.loc[has, b_col].to_numpy()
    return arr[a] if which == "first" else arr[b - 1]
# Add cols to DataFrame
for col in ["acc_first_s", "acc_last_s", "gyro_first_s", "gyro_last_s"]:
    windows[col] = np.nan
# Populate the data into the DataFrame - calls edge to look up the exact timestamps in arrary t
# Maps the window index rangs (acc_a to acc_b and gyro_a to gyro_b) to actual values and saves them into the new cols
windows.loc[has, "acc_first_s"] = edge("acc_a", "acc_b", t, "first")
windows.loc[has, "acc_last_s"] = edge("acc_a", "acc_b", t, "last")
windows.loc[has, "gyro_first_s"] = edge("gyro_a", "gyro_b", t, "first")
windows.loc[has, "gyro_last_s"] = edge("gyro_a", "gyro_b", t, "last")

# Resuidual = gyro minus accel (seconds)
windows["start_res_s"] = windows["gyro_first_s"] - windows["acc_first_s"]
windows["end_res_s"] = windows["gyro_last_s"] - windows["acc_last_s"]
windows["window_tol_ok"] = (windows["start_res_s"].abs() <= CLOCK_TOL_S) & (windows["end_res_s"].abs() <= CLOCK_TOL_S) 

# Trace to raw records - first/last raw line (Source_Line) of each sensor window in its source file
def trace_lines(w, lines, prefix):
    """First/last 1-based raw line of each sensor window (NA when the window has no samples)"""
    m = (w[f"{prefix}_n"] > 0).to_numpy()
    first = pd.Series(pd.NA, index = w.index, dtype = "Int64")
    last = first.copy()
    first[m] = lines[w.loc[m, f"{prefix}_a"].to_numpy()]
    last[m] = lines[w.loc[m, f"{prefix}_b"].to_numpy() - 1]
    return first, last

windows["acc_line_first"], windows["acc_line_last"] = trace_lines(windows, tw["Source_Line"].to_numpy(), "acc")
windows["gyro_line_first"], windows["gyro_line_last"] = trace_lines(windows, tw["Source_Line"].to_numpy(), "gyro")
acc_windows["acc_line_first"], acc_windows["acc_line_last"] = trace_lines(acc_windows, fw["Source_Line"].to_numpy(), "acc")

# Rejection reasons: every rule a window fails, separated by ";" ("" = accepted)
def add_reason(reason, bad, label):
    return reason.mask(bad, reason + ";" + label)

def rule_reasons(w, prefixes):
    reason = pd.Series("", index = w.index, dtype = object)
    for p in prefixes:
        reason = add_reason(reason, w[f"{p}_n"] < MIN_N, f"{p}_n<{MIN_N}")
        reason = add_reason(reason, w[f"{p}_span"] < MIN_SPAN_S, f"{p}_span<{MIN_SPAN_S}")
        reason = add_reason(reason, w[f"{p}_gap"] > MAX_GAP_S, f"{p}_gap>{MAX_GAP_S}")
    return reason

# A window is matched when both sensors pass the frozen rules AND its own start/end residuals are within CLOCK_TOL_S.
# The pair-level offsets (tol_ok, cell 16) are reported clock evidence, not a gate: a gyro that starts late only
# affects the first window of the pair, and that window is already rejected by its own residual.
windows["matched"] = windows["valid"] & windows["window_tol_ok"]
reason = rule_reasons(windows, ["acc", "gyro"])
reason = add_reason(reason, ~windows["window_tol_ok"], "window_residual_outside_tolerance")
windows["reason"] = reason.str.lstrip(";")
acc_windows["reason"] = rule_reasons(acc_windows, ["acc"]).str.lstrip(";")
assert windows["matched"].equals(windows["reason"].eq("")) and acc_windows["valid"].equals(acc_windows["reason"].eq(""))

windows["status"] = np.where(windows["matched"], "accepted", "rejected")
acc_windows["status"] = np.where(acc_windows["valid"], "accepted", "rejected")

# Residuals and unmatched pairs count
windows[["window_id", "pair_id", "Subject_ID", "Device", "Activity_Code",
         "t_start", "start_res_s", "end_res_s", "window_tol_ok", "tol_ok"]].to_csv("window_residual.csv", index = False)

residual_summary = windows[windows["valid"]].groupby("Device")[["start_res_s", "end_res_s"]].describe(percentiles=[.05, .5, .95]).T.round(4)
print("Residual on valid windows (s):")
print(residual_summary.to_string())
print("Window status by split and device:")
print(windows.groupby(["split", "Device"])["matched"].agg(windows="size", matched="sum", rate="mean").round(3).to_string())
print("Rejection reasons by device:")
print(windows[~windows["matched"]].groupby(["Device", "reason"]).size().to_string())
print("Accepted windows from pairs outside the segment-level tolerance (reported only):")
print(windows[windows["matched"] & ~windows["tol_ok"]].groupby("Device").size().to_string())

# Segment-level unmatched counts (accel segments with no gyro partner and vice versa) from the pairing step
print(f"Unmatched segments: accel w/o gyro = {len(unpaired_acc)}, gyro w/o accel = {len(unpaired_gyro)}")

PAIR_PASS_MIN = 0.95
tr = windows[(windows["split"] == "train") & windows["valid"]]
pair_rate = tr.groupby("Device")["matched"].mean()

print("Train pairing rate (matched/rules-valid)") 
print(pair_rate.round(3).to_string())

PAIRING_OK = bool(pair_rate.get("watch", 0) >= PAIR_PASS_MIN)
FEATURE_MODE = "watch_accel_gyro" if PAIRING_OK else "watch_accel_15_fallback"

print(f"PAIRING_OK (watch) = {PAIRING_OK}; FEATURE_MODE = {FEATURE_MODE}")

# Window manifest: EVERY candidate watch window of the chosen mode, with valid / status / reason and raw-source trace
manifest = (windows[windows["Device"] == "watch"] if PAIRING_OK else acc_windows).copy()
MANIFEST_COLS = ["window_id", "Subject_ID", "Device", "Activity_Code", "occ", "split", "t_start", "acc_seg", "gyro_seg",
                 "acc_source", "acc_line_first", "acc_line_last", "gyro_source", "gyro_line_first", "gyro_line_last",
                 "acc_n", "acc_span", "acc_gap", "gyro_n", "gyro_span", "gyro_gap", "start_res_s", "end_res_s",
                 "valid", "status", "reason"]
manifest[[c for c in MANIFEST_COLS if c in manifest.columns]].to_csv("window_manifest.csv", index = False)
print(f"window manifest ({FEATURE_MODE}): {(manifest['status'] == 'accepted').sum()} accepted, {(manifest['status'] == 'rejected').sum()} rejected")

# Exclusion log (rows / segments) and the exclusion audit by user and class for the chosen mode
exclusions = pd.concat(EXCLUDE, ignore_index = True).drop_duplicates()
exclusions = exclusions.astype({"Seg_ID": "Int64", "n_rows": "Int64", "Stream": str, "Activity_Code": str})
exclusions.to_csv("exclusions.csv", index = False)

used_streams = ["watch_accel", "watch_gyro"] if PAIRING_OK else ["watch_accel"]
used_levels = ["rows", "segment"] + (["segment_pairing"] if PAIRING_OK else [])
ex_used = exclusions[exclusions["Stream"].isin(used_streams) & exclusions["level"].isin(used_levels)]
rows_by_uc = ex_used.pivot_table(index = ["Subject_ID", "Activity_Code"], columns = "reason", values = "n_rows",
                                 aggfunc = "sum", fill_value = 0).add_prefix("rows_excluded:")
wins_by_uc = manifest.groupby(["Subject_ID", "Activity_Code", "status"]).size().unstack("status", fill_value = 0).add_prefix("windows_")
exclusion_audit = wins_by_uc.join(rows_by_uc, how = "outer").fillna(0).astype(int)
exclusion_audit.to_csv("exclusions_by_user_class.csv")

print(f"\nexclusion log: {len(exclusions)} entries")
print(exclusions.groupby(["level", "reason"])["n_rows"].agg(entries = "size", rows = "sum").to_string())
print("\nuser/class cells with rejected windows or excluded rows:")
display(exclusion_audit[(exclusion_audit.drop(columns = "windows_accepted", errors = "ignore") > 0).any(axis = 1)])
print("saved window_manifest.csv, window_residual.csv, exclusions.csv and exclusions_by_user_class.csv")

Residual on valid windows (s):
Device                  phone       watch
start_res_s count  15606.0000  15435.0000
            mean       0.0147      0.0037
            std        0.0723      0.0154
            min       -0.2307     -0.1045
            5%         0.0000      0.0000
            50%        0.0000      0.0000
            95%        0.0481      0.0495
            max        1.5822      0.4454
end_res_s   count  15606.0000  15435.0000
            mean      -0.0030     -0.0007
            std        0.0085      0.0038
            min       -0.0425     -0.0403
            5%        -0.0202      0.0000
            50%        0.0000      0.0000
            95%        0.0001      0.0000
            max        0.0839      0.0298
Window status by split and device:
              windows  matched   rate
split Device                         
eval  phone      3085     3045  0.987
      watch      3061     3060  1.000
test  phone      3095     3059  0.988
      watch      3050     3049

windows_accepted  windows_rejected  \
Subject_ID Activity_Code                                       
1602       S                            17                 1   
1607       C                            17                 1   
1629       A                            17                 0   
           B                            17                 0   
           C                            17                 0   
           D                            17                 0   
           E                            17                 0   
           F                            17                 0   
           G                            17                 0   
           H                            17                 0   
           I                            17                 0   
           J                            17                 0   
           K                            17                 0   
           L                            17                 0   
           M                            17                 0   
           O                            17                 0   
           P                            17                 0   
           Q                            17                 0   
           R                            17                 0   
           S                            17                 0   
1630       C                            17                 1   
1637       C                             0                 0   
1638       H                             0                 0   
           O                             0                 0   
1639       C                            16                 1   
           D                            16                 1   
1640       C                             0                 0   
           E                             0                 0   
1649       M                            11                 0   
           O                            12                 0   

                          rows_excluded:Label_Conflict_Shared_Timestamp  \
Subject_ID Activity_Code                                                  
1602       S                                                          0   
1607       C                                                          0   
1629       A                                                          0   
           B                                                          0   
           C                                                          0   
           D                                                          0   
           E                                                          0   
           F                                                          0   
           G                                                          0   
           H                                                          0   
           I                                                          0   
           J                                                          0   
           K                                                          0   
           L                                                          0   
           M                                                          0   
           O                                                          0   
           P                                                          0   
           Q                                                          0   
           R                                                          0   
           S                                                          0   
1630       C                                                          0   
1637       C                                                          0   
1638       H                                                          0   
           O                                                          0   
1639       C               

saved window_manifest.csv, window_residual.csv, exclusions.csv and exclusions_by_user_class.csv


In [25]:
# Watch feature extraction - explicit feature-column list, never "all numeric columns"

STATS = ["mean", "std", "min", "max", "rms"]
AXES = ["x", "y", "z"]
ACC_COLS = [f"acc_{a}_{s}" for a in AXES for s in STATS]
GYRO_COLS = [f"gyro_{a}_{s}" for a in AXES for s in STATS]
FEATURE_COLS = ACC_COLS + (GYRO_COLS if FEATURE_MODE == "watch_accel_gyro" else [])

def feats(xyz, a, b):
    w = xyz[a:b].astype("float64")
    # Order to match (x: mean, std, min, max, rms) then y then z to match ACC_COLS and GYRO_COLS
    return np.column_stack([w.mean(0), w.std(0), w.min(0), w.max(0), np.sqrt((w**2).mean(0))]).ravel()

TRACE_COLS = ["window_id", "pair_id", "Subject_ID", "Device", "Activity_Code", "occ", "split", "t_start",
              "acc_seg", "acc_source", "acc_line_first", "acc_line_last",
              "gyro_seg", "gyro_source", "gyro_line_first", "gyro_line_last"]

# Paired mode needs a matched window (both sensors valid + window residuals within tolerance); the fallback only needs its
# accel half to be valid, and uses the fallback windows built on every watch accel segment (cell 18), not just the paired ones
if PAIRING_OK:
    xyz = tw[["x", "y", "z"]].to_numpy()
    sel = windows[(windows["Device"] == "watch") & windows["matched"]]
    rows = [np.r_[feats(xyz, r.acc_a, r.acc_b), feats(xyz, r.gyro_a, r.gyro_b)] for r in sel.itertuples()]
else:
    xyz = fw[["x", "y", "z"]].to_numpy()
    sel = acc_windows[acc_windows["valid"]]
    rows = [feats(xyz, r.acc_a, r.acc_b) for r in sel.itertuples()]

features_watch = pd.concat([sel[[c for c in TRACE_COLS if c in sel.columns]],
                            pd.DataFrame(rows, columns = FEATURE_COLS, index = sel.index)], axis = 1)
features_watch["label"] = features_watch["Activity_Code"].astype(str).map(ACTIVITY)
features_watch["feature_mode"] = FEATURE_MODE

# Checks to make sure data is ready for model training w/ features
assert np.isfinite(features_watch[FEATURE_COLS].to_numpy()).all(), "non-finite feature values"
assert features_watch["window_id"].is_unique, "duplicate window IDs"
assert features_watch["label"].notna().all(), "unknown activity code"
assert features_watch["split"].notna().all(), "feature window from a user outside the frozen split"
assert features_watch.groupby("Subject_ID")["split"].nunique().max() == 1, "a participant appears in more than one split"
assert features_watch["acc_line_first"].notna().all(), "feature window without a raw-source trace"

# Class coverage of held-out users in the FINAL feature table (cell 19 checked it on rule-valid accel windows;
# the paired mode can still lose windows to the gyro rules or the window residual check)
held_out = features_watch[features_watch["split"].isin(["eval", "test"])]
have = held_out.groupby("Subject_ID")["Activity_Code"].agg(lambda s: set(s))
coverage_gaps = pd.DataFrame({"split": held_out.groupby("Subject_ID")["split"].first(),
                              "missing_classes": ["".join(c for c in REQUIRED_CLASSES if c not in h) for h in have]})
coverage_gaps = coverage_gaps[coverage_gaps["missing_classes"] != ""]
if len(coverage_gaps):
    print(f"WARNING - held-out users missing a class in the {FEATURE_MODE} feature table (see window_manifest.csv reasons):")
    display(coverage_gaps)

features_watch.to_parquet("features_watch.parquet", index = False)
pd.Series(FEATURE_COLS, name = "features").to_csv("feature_columns.csv", index = False)
print(features_watch.shape, "| mode:", FEATURE_MODE)
print(features_watch.groupby("split")["Subject_ID"].nunique().to_string())
print(features_watch.groupby(["split", "Activity_Code"]).size().unstack("split").to_string())

(15430, 48) | mode: watch_accel_gyro
split
eval     10
test     10
train    31
split          eval  test  train
Activity_Code                   
A               170   170    529
B               170   170    512
C               170   170    475
D               170   170    529
E               170   170    511
F               170   170    513
G               170   170    528
H               170   170    511
I               170   170    529
J               170   170    511
K               170   170    528
L               170   170    511
M               170   164    528
O               170   165    494
P               170   170    529
Q               170   170    528
R               170   170    527
S               170   170    528


In [26]:
# Audit the cross-device clock. Phone and watch timestamps are compared on their raw int64 clocks
# One row per (Subject, Device, Activity): the span of all kept segments of that device for the activity.
# No occurrence numbers and no overlap search are used for matching - the overlap is exactly what is being tested.

dev = (seg.reset_index().groupby(["Subject_ID", "Device", "Activity_Code"])
          .agg(t0 = ("t0", "min"), t1 = ("t1", "max"), n_segments = ("Seg_ID", "size")).reset_index())

phone = dev[dev["Device"] == "phone"].drop(columns = "Device")
watch = dev[dev["Device"] == "watch"].drop(columns = "Device")
xd = phone.merge(watch, on = ["Subject_ID", "Activity_Code"], suffixes = ("_phone", "_watch"))

xd["start_off_s"] = (xd["t0_phone"] - xd["t0_watch"]) / 1e9
xd["end_off_s"] = (xd["t1_phone"] - xd["t1_watch"]) / 1e9

ov = (np.minimum(xd["t1_phone"], xd["t1_watch"]) - np.maximum(xd["t0_phone"], xd["t0_watch"])) / 1e9
sh = np.minimum(xd["t1_phone"] - xd["t0_phone"], xd["t1_watch"] - xd["t0_watch"]) / 1e9

xd["overlaps"] = ov >= MIN_OVERLAP_FRAC * sh
xd["direct_ok"] = xd["overlaps"] & (xd["start_off_s"].abs() <= CLOCK_TOL_S) & (xd["end_off_s"].abs() <= CLOCK_TOL_S)

print(f"Phone/Watch activity pairs: {len(xd)} | Overlapping on Raw Clock: {int(xd['overlaps'].sum())} | Within {CLOCK_TOL_S}s: {int(xd['direct_ok'].sum())}")
print(xd["start_off_s"].describe(percentiles=[.05, .5, .95]).round(2).to_string())

# Is there a stable per-participant offset even if clocks are not shared? This is only a CANDIDATE:
# the offset is never applied, and independently reset clocks with a small spread are not shared-clock evidence.
by_subject = xd.groupby("Subject_ID")["start_off_s"].agg(median = "median", spread = "std", n = "size")
by_subject["stable"] = by_subject["spread"] <= CLOCK_TOL_S

print(f"Participant with a stable phone-watch offset (std <= {CLOCK_TOL_S}s): {int(by_subject['stable'].sum())} / {len(by_subject)}")

SHARED_CLOCK_MIN = 0.95
direct_share = float(xd["direct_ok"].mean()) if len(xd) else 0.0
stable_share = float(by_subject["stable"].mean()) if len(by_subject) else 0.0
# Only direct shared-clock evidence can unlock fusion; a stable offset is reported as a candidate and NOT used
CROSS_DEVICE = "shared_clock" if direct_share >= SHARED_CLOCK_MIN else "unavailable"
OFFSET_CANDIDATE = "candidate_not_used" if stable_share >= SHARED_CLOCK_MIN else "none"
FUSE_PHONE_WATCH = False # No phone/watch fusion this week, whatever the audit says - fusion needs its own justified matching step

print(f"direct share = {direct_share:.3f} | stable per-subject offset share = {stable_share:.3f} ({OFFSET_CANDIDATE}) | "
      f"CROSS_DEVICE = {CROSS_DEVICE} | phone/watch fusion this week: {FUSE_PHONE_WATCH}")

xd.to_csv("cross_device_audit.csv", index = False)

# Alignment audit write-up: clock evidence, overlap, matching tolerance, rejection counts and both decisions
n_acc = (manifest["status"] == "accepted").sum(); n_rej = (manifest["status"] == "rejected").sum()
audit_lines = [
    "# Alignment audit (Week 2)", "",
    "## Accelerometer / gyroscope on the same device",
    f"- Pairing: time overlap inside (Subject, Device, Activity), one-to-one, largest overlap first; overlap must cover >= {MIN_OVERLAP_FRAC:.0%} of the shorter segment. Labels, occurrence numbers and row numbers are not used as evidence.",
    f"- Segments: {len(acc)} accel and {len(gyr)} gyro -> {len(pairs)} pairs. Unpaired: {len(unpaired_acc)} accel, {len(unpaired_gyro)} gyro (listed in exclusions.csv).",
    f"- Segment-level offsets within {CLOCK_TOL_S} s (reported evidence, not a gate): {int(pairs['tol_ok'].sum())} / {len(pairs)} pairs.",
    f"- Window gate: both sensors pass the frozen rules and the window's own start/end residuals are within {CLOCK_TOL_S} s.",
    "", "Segment start/end offsets, gyro minus accel (s):", "```", clock_evidence.to_string(), "```", "",
    "Window-level residuals on rule-valid windows, gyro minus accel (s):", "```", residual_summary.to_string(), "```", "",
    "- Train pairing rate (matched / rule-valid windows): " + ", ".join(f"{d} {r:.3f}" for d, r in pair_rate.items()),
    f"- Threshold {PAIR_PASS_MIN:.0%} on the watch -> PAIRING_OK = {PAIRING_OK}; FEATURE_MODE = {FEATURE_MODE}.",
    f"- Watch windows in this mode: {n_acc} accepted, {n_rej} rejected (per-window reasons in window_manifest.csv; residuals in window_residual.csv).",
    "",
    "## Phone / watch (cross-device)",
    f"- {len(xd)} (subject, activity) spans compared on the raw clocks: {int(xd['overlaps'].sum())} overlap, {int(xd['direct_ok'].sum())} agree within {CLOCK_TOL_S} s (share {direct_share:.3f}; {SHARED_CLOCK_MIN:.0%} needed).",
    f"- Stable per-subject offset (std <= {CLOCK_TOL_S} s): {int(by_subject['stable'].sum())} / {len(by_subject)} subjects -> {OFFSET_CANDIDATE}. A stable offset is not shared-clock evidence and is never applied.",
    f"- Decision: CROSS_DEVICE = {CROSS_DEVICE}; phone/watch fusion this week: {FUSE_PHONE_WATCH}. Details in cross_device_audit.csv.",
    "- Matching labels, row numbers or independently reset timestamps are not accepted as alignment evidence.",
]
with open("alignment_audit.md", "w") as f:
    f.write("\n".join(audit_lines) + "\n")
print("saved cross_device_audit.csv and alignment_audit.md")

Phone/Watch activity pairs: 908 | Overlapping on Raw Clock: 0 | Within 0.25s: 0
count        908.00
mean     -116834.11
std       595578.19
min     -2068451.40
5%      -1631338.40
50%         -551.68
95%       569008.97
max      1125629.43
Participant with a stable phone-watch offset (std <= 0.25s): 23 / 51
direct share = 0.000 | stable per-subject offset share = 0.451 (none) | CROSS_DEVICE = unavailable | phone/watch fusion this week: False
saved cross_device_audit.csv and alignment_audit.md
